# Clase 8 — Casos de uso reales: la metodología completa en acción

**Duración estimada: 120 minutos**

## Objetivos de aprendizaje

- Recorrer de principio a fin un caso real (reingresos hospitalarios) por las 10 etapas de la metodología.
- Ejecutar un pipeline completo (carga → limpieza → modelado → evaluación) sobre un dataset real.
- Aplicar el pipeline aprendido a un subconjunto de datos nuevo de forma autónoma.

Este es el último notebook de teoría del curso. No introduce conceptos nuevos: reúne todo lo que ya viste en las clases 1 a 7 en dos cosas: (1) un recorrido condensado del caso completo de reingresos por insuficiencia cardíaca congestiva (CHF) por las 10 etapas de la metodología de Rollins, y (2) el pipeline de código completo — carga, limpieza, modelado y evaluación — sobre el dataset de recetas, encadenado de principio a fin en un solo lugar, tal como lo ejecutaría un data scientist en un proyecto propio.

## 1. El caso CHF completo, las 10 etapas

Repaso ejecutivo — ya viste cada una de estas etapas en detalle en capítulos anteriores del manual. Aquí las ves todas seguidas, tal como ocurrieron en el proyecto real.

| # | Etapa | Qué pasó |
|---|---|---|
| 1 | **Business Understanding** | Una aseguradora de salud, ante la caída de financiación pública para reingresos, se une a IBM. Pregunta: ¿cómo asignar un presupuesto de salud limitado para maximizar calidad de atención? Se descubre que ~30% de pacientes rehabilitados reingresan en 1 año (50% en 5 años), y que CHF encabeza la lista. Salen 4 requisitos de negocio de un workshop. |
| 2 | **Analytic Approach** | Se elige clasificación por árbol de decisión: fácil de leer para personal clínico, da predicción + probabilidad. |
| 3 | **Data Requirements** | Cohorte definido con 3 criterios de inclusión (admisión como interno, diagnóstico principal de CHF durante 1 año, 6 meses de cobertura previa) y exclusión de comorbilidades graves. Formato objetivo: un registro por paciente. |
| 4 | **Data Collection** | Fuentes demográficas, clínicas, de cobertura, reclamaciones y farmacéuticas. Se aplaza el dato farmacéutico (no disponible) — decisión correcta, no hizo falta al final. |
| 5 | **Data Understanding** | Estadística descriptiva revela que la definición inicial de "admisión por CHF" no capturaba todos los casos esperados clínicamente. Se vuelve a Data Collection. |
| 6 | **Data Preparation** | Se define CHF con precisión (códigos de diagnóstico + guía clínica), se fija la ventana de reingreso en 30 días, se agregan miles de registros transaccionales a un registro por paciente. Cohorte final: **2.343 pacientes**. |
| 7 | **Modeling** | Primer árbol: 85% accuracy global pero solo 45% de aciertos en "sí". Se ajusta el costo relativo de clasificación errónea: 1:1 (85%/45%), 9:1 (49%/97%, demasiados falsos positivos), 4:1 (81% global, 68% sensibilidad, 85% especificidad) — mejor equilibrio. |
| 8 | **Evaluation** | La curva ROC confirma que el modelo 4:1 da la máxima separación respecto a la línea base — modelo óptimo. |
| 9 | **Deployment** | Se traduce a una app para tablet/navegador con evaluación de riesgo casi en tiempo real, con formación del personal y colaboración con TI/DBAs. |
| 10 | **Feedback** | Ejecutivos de gestión clínica revisan resultados, se hace seguimiento de pacientes con intervención, se comparan tasas de reingreso antes/después (sin grupo de control, por ética) al cabo de un año, y se refina el modelo — incluyendo la posibilidad de incorporar por fin el dato farmacéutico aplazado. |

**Resultado:** médicos que pudieron dar mejor atención a sus pacientes incorporando información basada en datos, casi en tiempo real, a sus decisiones clínicas.

## 2. El caso hermano: Cognos y diabetes juvenil

La misma metodología — y la misma técnica de árbol de decisión — se reutilizó en un problema clínico distinto: predecir el riesgo de hospitalización de pacientes con diabetes juvenil. Se desplegó como una aplicación interactiva con **Cognos**, con tres niveles de vista:

- **Vista nacional**: mapa interactivo del riesgo de hospitalización en todo el país.
- **Vista por subgrupo**: informe resumido del riesgo dentro de un nodo concreto del árbol de decisión.
- **Vista individual**: informe detallado del riesgo y el historial de un paciente concreto.

Esto demuestra que la metodología de Rollins no es una receta que solo sirve para el caso CHF: es un proceso reutilizable para cualquier problema, incluso con la misma técnica de modelado.

## 3. El pipeline completo: recetas, de principio a fin

A partir de aquí, todo es código real que ya conoces de las clases 3, 4 y 5. Vas a verlo encadenado en un solo bloque continuo, con comentarios que marcan a qué etapa de la metodología corresponde cada parte — exactamente como lo ejecutaría un data scientist en su propio proyecto, sin pausas entre etapas.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import re
import random
from sklearn import tree
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import itertools

### Etapa: Data Collection

Descarga el CSV directamente desde el servidor de IBM y lo carga como DataFrame — exactamente como en la clase 3.

In [ ]:
# Etapa: Data Collection
recipes = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-DS0103EN-SkillsNetwork/labs/Module%202/recipes.csv"
)

print("Data read into dataframe!", recipes.shape)

### Etapa: Data Preparation

El bloque de limpieza completo que viste en la clase 4: arreglar el nombre de columna, poner todo en minúsculas, unificar nombres de cocina inconsistentes, eliminar cocinas con menos de 50 recetas, y convertir "Yes"/"No" a 1/0.

In [ ]:
# Etapa: Data Preparation

# La primera columna trae el nombre de la cocina, pero mal etiquetada como "country"
column_names = recipes.columns.values
column_names[0] = "cuisine"
recipes.columns = column_names

# Minúsculas, para que "Mexican" y "mexican" no cuenten como cocinas distintas
recipes["cuisine"] = recipes["cuisine"].str.lower()

# Corrige nombres de cocina inconsistentes
recipes.loc[recipes["cuisine"] == "korea", "cuisine"] = "korean"
recipes.loc[recipes["cuisine"] == "japan", "cuisine"] = "japanese"
recipes.loc[recipes["cuisine"] == "china", "cuisine"] = "chinese"
recipes.loc[recipes["cuisine"] == "thailand", "cuisine"] = "thai"
recipes.loc[recipes["cuisine"] == "india", "cuisine"] = "indian"

# Elimina cocinas con menos de 50 recetas: no hay suficientes ejemplos para aprender un patrón fiable
recipes_counts = recipes["cuisine"].value_counts()
cuisines_indices = recipes_counts > 50
cuisines_to_keep = list(np.array(recipes_counts.index.values)[np.array(cuisines_indices)])
recipes = recipes.loc[recipes["cuisine"].isin(cuisines_to_keep)]

# Convierte "Yes"/"No" a 1/0, el formato que scikit-learn necesita
recipes = recipes.replace(to_replace="Yes", value=1)
recipes = recipes.replace(to_replace="No", value=0)

print("Dataset limpio y listo:", recipes.shape)

### Etapa: Modeling

Subconjunto de 5 cocinas (coreana, japonesa, china, tailandesa e india), división train/test con `sample_n = 30` y `random.seed(1234)` para que el resultado sea reproducible, y entrenamiento de un `DecisionTreeClassifier` con `max_depth=15` — igual que en la clase 5.

In [ ]:
# Etapa: Modeling
bamboo = recipes[recipes.cuisine.isin(["korean", "japanese", "chinese", "thai", "indian"])]

sample_n = 30
random.seed(1234)  # fija la semilla aleatoria para que el resultado sea reproducible

# Conjunto de prueba: 30 recetas de cada cocina
bamboo_test = bamboo.groupby("cuisine", group_keys=False).apply(lambda x: x.sample(sample_n))
bamboo_test_ingredients = bamboo_test.iloc[:, 1:]
bamboo_test_cuisines = bamboo_test["cuisine"]

# Conjunto de entrenamiento: todo lo que NO quedó en el conjunto de prueba
bamboo_train_index = bamboo.index.isin(bamboo_test.index)
bamboo_train = bamboo[~bamboo_train_index]
bamboo_train_ingredients = bamboo_train.iloc[:, 1:]
bamboo_train_cuisines = bamboo_train["cuisine"]

bamboo_train_tree = tree.DecisionTreeClassifier(max_depth=15)
bamboo_train_tree.fit(bamboo_train_ingredients, bamboo_train_cuisines)

print("Decision tree model saved to bamboo_train_tree!")

Puedes visualizar el árbol entrenado con `plot_tree` de scikit-learn, exactamente igual que en la clase 5 — no usamos `graphviz`, porque no funciona en Google Colab sin instalar dependencias extra.

In [ ]:
# Visualización del árbol con plot_tree (no graphviz), igual que en la clase 5
plt.figure(figsize=(20, 10))
tree.plot_tree(
    bamboo_train_tree,
    feature_names=list(bamboo_train_ingredients.columns.values),
    class_names=sorted(bamboo_train_cuisines.unique()),
    filled=True,
    fontsize=6,
    max_depth=3,  # solo mostramos los primeros niveles para que sea legible
)
plt.show()

In [ ]:
# .predict() usa el modelo ya entrenado para predecir la cocina
# de recetas que el modelo NUNCA vio durante el entrenamiento
bamboo_pred_cuisines = bamboo_train_tree.predict(bamboo_test_ingredients)

bamboo_pred_cuisines[:10]

### Etapa: Evaluation

Matriz de confusión (filas = cocina real, columnas = cocina predicha) y accuracy global — la misma lectura que hiciste con la tabla de modelos del caso CHF: nunca te quedes con un solo número.

In [ ]:
# Etapa: Evaluation
test_cuisines = np.unique(bamboo_test_cuisines)
bamboo_confusion_matrix = confusion_matrix(bamboo_test_cuisines, bamboo_pred_cuisines, labels=test_cuisines)

title = "Bamboo Confusion Matrix"
cmap = plt.cm.Blues

plt.figure(figsize=(8, 6))
bamboo_confusion_matrix_pct = (
    bamboo_confusion_matrix.astype("float") / bamboo_confusion_matrix.sum(axis=1)[:, np.newaxis]
) * 100

plt.imshow(bamboo_confusion_matrix_pct, interpolation="nearest", cmap=cmap)
plt.title(title)
plt.colorbar()
tick_marks = np.arange(len(test_cuisines))
plt.xticks(tick_marks, test_cuisines)
plt.yticks(tick_marks, test_cuisines)

fmt = ".2f"
thresh = bamboo_confusion_matrix_pct.max() / 2.0
for i, j in itertools.product(range(bamboo_confusion_matrix_pct.shape[0]), range(bamboo_confusion_matrix_pct.shape[1])):
    plt.text(
        j, i, format(bamboo_confusion_matrix_pct[i, j], fmt),
        horizontalalignment="center",
        color="white" if bamboo_confusion_matrix_pct[i, j] > thresh else "black",
    )

plt.tight_layout()
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.show()

In [ ]:
# Accuracy global: proporción de predicciones correctas sobre el total de recetas de prueba
accuracy_global = accuracy_score(bamboo_test_cuisines, bamboo_pred_cuisines)
print(f"Accuracy global del modelo: {accuracy_global:.2%}")

## Resumen: el mismo viaje, dos formas de recorrerlo

Fíjate en la simetría entre lo que acabas de ejecutar y el caso CHF que viste al principio de este notebook:

| Etapa de la metodología | Caso CHF | Caso recetas (este notebook) |
|---|---|---|
| Data Collection | Fuentes demográficas, clínicas, de cobertura, reclamaciones, farmacéuticas | `pd.read_csv()` desde el servidor de IBM |
| Data Preparation | Definir CHF con precisión, ventana de 30 días, agregar a 1 registro/paciente (2.343 pacientes) | Nombre de columna, minúsculas, consistencia, filtro <50, Yes/No → 1/0 |
| Modeling | Árbol de decisión, ajuste de costo relativo de clasificación errónea (1:1, 9:1, 4:1) | `DecisionTreeClassifier(max_depth=15)` sobre 5 cocinas |
| Evaluation | Curva ROC — el modelo 4:1 da la máxima separación | Matriz de confusión + accuracy global |

El caso CHF no tiene datos públicos que puedas ejecutar, pero la lógica que acabas de correr sobre recetas es exactamente la misma que corrió ese equipo de IBM sobre datos hospitalarios reales. Ahora ve al notebook de práctica de esta clase y repite este mismo pipeline tú solo, sobre un subconjunto de cocinas europeas — con menos pistas que en cualquier práctica anterior del curso.